# 04 · Model preparation — PSMA / GCPII (Q04609)

**Milestone 5.** This notebook prepares a full-length, membrane-anchored PSMA dimer for MD.
Design and every decision: [`docs/M5_MODEL_PREPARATION.md`](../docs/M5_MODEL_PREPARATION.md).

The notebook builds a **hybrid model** and prepares it:

1. **Base.** The experimental **9S7U** ectodomain dimer (biological assembly; closed entrance lid) supplies every residue it models, plus its Zn²⁺, Ca²⁺ and Cl⁻ ions, the waters around the Zn site and its core N-glycans.
2. **AlphaFold2 anchor.** The milestone 4 **AlphaFold2-multimer** model `CF:dimer:rank_001` supplies:
   - the cytoplasmic tail, TM helix and stalk, placed in the global dimer frame so the two TM helices keep their predicted arrangement, with the graft point chosen for the best peptide bond;
   - internal gaps, locally fitted on their flanking residues;
   - missing side-chain atoms;
   - the wild-type residue wherever 9S7U carries a substitution.
3. **Zn site.** The coordination geometry is checked against the configured ligands. The **MD treatment is your decision** (`ZN_MD_TREATMENT` below), and until you make it, the readiness table stays FAIL.
4. **N-glycans.** A core GlcNAc is placed at every sequon observed glycosylated in the PDB. It is kept from 9S7U where present, otherwise transplanted from another entry, and rejected if it would clash.
5. **Protonation.** PROPKA at pH 7.4 assigns the states, with these rules:
   - Zn-ligand His tautomers come from the coordination geometry;
   - Zn-ligand Asp/Glu are fixed as deprotonated;
   - disulfides are detected;
   - **alternative microstates** are written for residues within 1 pH unit of their pKa, and for the catalytic E424.
6. **Membrane.** The C2-constrained membrane fit from milestone 4 is applied to the hybrid, and membrane-oriented PDB files are written.
7. **Provenance.** Every step is recorded with input and output hashes. **Original files are never modified**, and a critical check confirms it.

Outputs go to `<ACC>_model_preparation.xlsx`, `preparation_tables/` and `model_preparation/hybrid_9S7U_AF2/`.

> **Runtime:** CPU is enough. Notebook 03 must have run first, because this notebook needs its AlphaFold2 dimer in the cache.

## 0 · Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = "https://github.com/victorolatunde23-dev/psma_structural_modelling.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("psma_structural_modelling").exists() and not Path("src/psma_pipeline").exists():
    subprocess.run(["git", "clone", "-q", REPO], check=True)
if Path("psma_structural_modelling").exists():
    os.chdir("psma_structural_modelling")
if Path("../src/psma_pipeline").exists():
    os.chdir("..")
if IN_COLAB:
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[membrane]", "propka"], check=True)
sys.path.insert(0, str(Path("src").resolve()))
for _m in [m for m in list(sys.modules) if m == "psma_pipeline" or m.startswith("psma_pipeline.")]:
    del sys.modules[_m]
import psma_pipeline
print("project root:", Path.cwd(), "| psma_pipeline", psma_pipeline.__version__)

## 1 · Configuration, your decisions, and earlier results

| Setting | Your choice | Where it is set |
|---|---|---|
| Template | **9S7U** (closed entrance lid) | `preparation.template` |
| Membrane anchor | **CF:dimer:rank_001** (AlphaFold2-multimer, notebook 03) | `preparation.anchor_model` |
| Glycans | **core GlcNAc where observed** | `preparation.glycans.strategy` |
| pH | **7.4** | `preparation.protonation.ph` |
| Zn MD treatment | **decide later** | `ZN_MD_TREATMENT` below: `None`, `"nonbonded_12_6_4"`, `"bonded_mcpb"` or `"cationic_dummy"` |

> **Decision: hybrid, not pure prediction.** Every residue, ion and water that 9S7U resolves is experimental. AlphaFold2 fills only what the experiment lacks: the membrane anchor, gaps and missing atoms. Every atom records its source (*ResidueSources*).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from psma_pipeline import load_config, setup_logging
from psma_pipeline.colab import storage_overrides
from psma_pipeline.assessment import load_survey_tables
from psma_pipeline.model_preparation import run_preparation, save_preparation
from psma_pipeline.sanity import critical_failures

setup_logging()
USE_DRIVE = True
ZN_MD_TREATMENT = None      # None (decide later) | "nonbonded_12_6_4" | "bonded_mcpb" | "cationic_dummy"
cfg = load_config(overrides={**(storage_overrides(USE_DRIVE) or {}),
                             "preparation": {"zn_md_treatment": ZN_MD_TREATMENT}})
SURVEY = cfg.output_file("survey_workbook")
ALPHAFOLD = cfg.output_file("alphafold_workbook")
assert Path(SURVEY).exists(), f"{SURVEY} not found: run notebook 01 first"
survey = load_survey_tables(SURVEY)
alphafold = load_survey_tables(ALPHAFOLD) if Path(ALPHAFOLD).exists() else None
print("survey:", SURVEY)
print("AlphaFold/membrane workbook:", ALPHAFOLD if alphafold else "not found (UniProt topology used)")
print("template:", cfg.get("preparation.template"), "| anchor:", cfg.get("preparation.anchor_model"),
      "| Zn treatment:", cfg.get("preparation.zn_md_treatment") or "not chosen")

## A · Unit tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-1500:])
assert r.returncode == 0, "Unit tests failed: fix before preparing the model."

## B · Run

> **Gate.** A critical failure stops the notebook:
> - `numbering_identity`: each hybrid chain must equal UniProt, residues 1–750;
> - `originals_unchanged`: the template and AlphaFold2 files must have the same hashes before and after the run;
> - `zn_site_complete`: two Zn per protomer, coordinated by the configured ligands;
> - `provenance_complete`: every output file must be recorded in `provenance.json`.

In [ ]:
res = run_preparation(cfg, survey, alphafold)
T = res.tables
checks = T["PrepChecks"]
display(checks)
crit = critical_failures(checks)
assert crit.empty, "critical preparation check failed: see PrepChecks"
display(T["PrepSummary"])
display(res.issues.to_frame())

## C · The hybrid: where every residue comes from

- **HybridSegments:** contiguous runs per source, with the fit used to place each AlphaFold2 segment.
- **Junctions:** the peptide C–N distance at every graft point (ideal ≈ 1.33 Å). Anything above 2.0 Å is flagged; Milestone 6 minimisation resolves small strain.
- **Repairs:** substitutions reverted to UniProt, and side chains completed from AlphaFold2.
- **TemplateHetero:** each 9S7U hetero group, with whether it was kept or removed.

In [ ]:
display(T["HybridSegments"])
display(T["Junctions"])
display(T["Repairs"])
display(T["TemplateHetero"])
src = T["ResidueSources"]
fig, ax = plt.subplots(figsize=(13, 1.8))
colors = {"experimental": "#7f7f7f", "membrane anchor": "#ff7f0e", "internal gap": "#d62728",
          "C-terminal tail": "#9467bd", "reverted": "#2ca02c", "side-chain": "#1f77b4"}
for y, (ch, g) in enumerate(src.groupby("Chain")):
    for u, s in zip(g["UniProt number"], g["Source"]):
        key = next((k for k in ("membrane anchor", "internal gap", "C-terminal tail", "reverted", "side-chain") if k in s),
                   "experimental")
        ax.barh(y, 1, left=u - 0.5, color=colors[key], height=0.7)
ax.set(yticks=range(src["Chain"].nunique()), yticklabels=sorted(src["Chain"].unique()), xlabel="UniProt residue",
       title="Residue sources of the hybrid")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=k) for k, c in colors.items()], ncol=6, fontsize=8, loc="upper center",
          bbox_to_anchor=(0.5, -0.45), frameon=False)
plt.tight_layout(); plt.show()

## D · Binuclear Zn site and the MD treatment decision

The Zn ions are experimental, from 9S7U. Each one is checked against the configured ligands:
- Zn1: H377, D387, D453;
- Zn2: D387, E425, H553.

Distances must fall within the configured range of 1.8–2.6 Å.

**Your decision (Milestone 6 needs it):**

| Treatment | In short |
|---|---|
| `nonbonded_12_6_4` | simplest; geometry not enforced |
| `bonded_mcpb` | geometry fixed by bonds; needs AmberTools and a QM calculation |
| `cationic_dummy` | in between |

The bridging water or hydroxide is decided together with the treatment.

In [ ]:
display(T["ZnSite"])
display(T["ZnContacts"])
display(T["MetalTreatmentOptions"])

## E · N-glycans (core GlcNAc where observed)

Crystallographic glycans are fragments, so only the **first GlcNAc** is placed, as a minimal, experimentally supported glycosylation.
- Sites never observed glycosylated are left unglycosylated, and listed.
- A transplanted sugar is rejected if any heavy atom comes within 2.0 Å of the protein.

In [ ]:
display(T["SequonObservations"])
display(T["Glycans"].pivot_table(index="Asn (UniProt)", columns="Chain", values="Decision", aggfunc="first"))
display(T["Glycans"][T["Glycans"]["Decision"] == "core GlcNAc"][["Chain", "Asn (UniProt)", "Source", "Note"]])

## F · Protonation at pH 7.4 and alternative microstates

States are written as AMBER residue names (HID/HIE/HIP, ASH, GLH, LYN, CYX), with heavy atoms only. Hydrogens are added by the system builder in Milestone 6.

The **default microstate** is the most likely state of every residue. Each **alternative** changes one residue:
- residues with |pKa − pH| ≤ 1;
- **E424**, the catalytic acid/base, always.

If a result depends on a residue's state, run it in both.

In [ ]:
pt = T["Protonation"]
display(pt[pt["State"].isin(["HID", "HIE", "HIP", "ASH", "GLH", "LYN", "CYX", "CYM"]) | pt["Alternative state"].notna()])
display(T["Microstates"])

## G · Membrane placement of the hybrid

In [ ]:
display(T["HybridMembrane"].T)
try:
    import py3Dmol
    f = res.files.get("membrane_oriented")
    v = py3Dmol.view(width=860, height=600)
    v.addModel(open(f).read(), "pdb")
    v.setStyle({"model": 0}, {"cartoon": {"color": "#bbbbbb"}})
    for ch, g in T["ResidueSources"].groupby("Chain"):
        for s, col in (("membrane anchor", "#ff7f0e"), ("internal gap", "#d62728"), ("reverted", "#2ca02c")):
            resi = g.loc[g["Source"].str.contains(s), "UniProt number"].tolist()
            if resi:
                v.setStyle({"model": 0, "chain": ch, "resi": resi}, {"cartoon": {"color": col}})
    v.addStyle({"model": 0, "resn": "ZN"}, {"sphere": {"color": "#7d7d7d", "radius": 1.2}})
    v.addStyle({"model": 0, "resn": ["NAG", "NDG"]}, {"stick": {"colorscheme": "greenCarbon"}})
    v.addModel(open(res.files["membrane_planes"]).read(), "pdb")
    v.setStyle({"model": 1}, {"sphere": {"radius": 0.5, "color": "#9ecae9", "opacity": 0.5}})
    v.rotate(-90, "x"); v.zoomTo({"model": 0}); v.show()
    print("grey = 9S7U · orange = AF2 membrane anchor · red = AF2 gap · green = reverted substitution · "
          "spheres = Zn · sticks = core GlcNAc")
except ImportError:
    print("py3Dmol not installed: open hybrid_membrane_oriented.pdb in PyMOL / ChimeraX")

## H · Readiness for MD

**The overall status stays FAIL in this milestone.** The prepared model is ready for system building, but not for MD. What remains:
- choose the Zn treatment;
- choose the bridging water or hydroxide;
- choose the lipid composition;
- in Milestone 6: the force field, parameters for Zn, glycans and lipids, hydrogens, solvation and minimisation.

In [ ]:
display(T["PrepReadiness"])

## I · Save

In [ ]:
paths = save_preparation(cfg, res)
for k, v in paths.items():
    print(f"{k}: {v}")

## Milestone 5 checklist

| Component | Status |
|---|---|
| Hybrid model (9S7U ectodomain dimer + AlphaFold2 membrane anchor), per-atom provenance | ✔ |
| Repair assessment: gaps, missing side-chain atoms, substitutions reverted, junction geometry, clashes | ✔ |
| Binuclear Zn site from experiment; coordination checks; MD-treatment options (user choice) | ✔ |
| Ions (Ca²⁺, Cl⁻) and Zn-shell waters kept; ligands/additives removed and listed | ✔ |
| N-glycans: core GlcNAc where observed (template or transplanted, clash-checked) | ✔ |
| Protonation (PROPKA, pH 7.4), Zn-ligand His tautomers, disulfides, alternative microstates | ✔ |
| Membrane placement of the hybrid; membrane-oriented model and microstate files | ✔ |
| Originals never overwritten (hash check), provenance chain, readiness (never READY) | ✔ |